## Verification Script of "Accelerated Concatenation of Stepsize Schedules for Smooth Strongly Convex Gradient Descent"

This script provides verifications of some complicated equations in the paper.

**Notations** 
| Variable | Definition  |
| --- | --- |
| $q$ | $\mu/L$ |
| $s $ | $\sigma_q(h )$ |
| $s_a$ | $\sigma_q(h_a)$ |
| $s_b$ | $\sigma_q(h_b)$ |
| $s_c$ | $\sigma_q(h_c)$ |
| $a$ | $\alpha$ |
| $b$ | $\beta$ |
| `eub`| $\mathbf{1}^\top u_b$ |
| `Cp` | $\mathcal C^{\mathrm p}$ |
| `Cf` | $\mathcal C^{\mathrm f}$ |

In [1]:
from sympy import symbols, QQ, Rational, sqrt, resultant

def xgf(i):
    return symbols('x_{%s} g_{%s} f_{%s}'%(i,i,i))

x0, g0, f0 = xgf(0); xn, gn, fn = xgf('n'); xm, gm, fm = xgf('m'); xs, gs, fs = xgf(r'\star')

q, mu, L, h, u, v, w, x, y, z, t, r = symbols('q mu L h u v w x y z t r')
P1, P2, R1, R2, s, sa, sb, sc, eub, a, b = symbols('P_{1} P_{2} R_{1} R_{2} s s_{a} s_{b} s_{c} 1^{T}u_{b} alpha beta')

def Q(i, j, mu=mu, L=L):
    xi, gi, fi = xgf(i)
    xj, gj, fj = xgf(j)
    return fi - fj + gi*(xj - xi) + (gi - gj)**2/(2*L) + mu/2/(1 - mu/L)*(xi - gi/L - xj + gj/L)**2

Qns = Q('n',r'\star',q*L,L).subs(gs, 0)
Qms = Q('m',r'\star',q*L,L).subs(gs, 0)
Qsn = Q(r'\star','n',q*L,L).subs(gs, 0)
Qsm = Q(r'\star','m',q*L,L).subs(gs, 0)
Qmn = Q('m','n',q*L,L)
Qnm = Q('n','m',q*L,L)

def simplify(expr):
    # faster than calling sympy.simplify because
    # it works on a rational function field
    K = QQ[expr.free_symbols].get_field()
    return K.to_sympy(K.from_sympy(expr))

### Section 2. Preliminaries and Notation

In [2]:
# Lemma 2.5 (ii)
xi, gi, fi = xgf('i'); xj, gj, fj = xgf('j')
lhs = Q('i',r'\star') + Q(r'\star','j') - 1/L/(1 - q)*(gj - L*(xj - xs))*(gi - mu*(xi - xs))

diff = (lhs - Q('i','j')).subs(gs, 0).subs(mu, q*L)
print(simplify(diff))

0


### Section 3.1. Schedule Classes and Performance Guarantees

In [3]:
# Theorem 3.3

lhs = mu/2*x**2 + (L-mu)*r*x - (L-mu)/2*r**2
mid = L/2*(1 + 2*s + q*s**2)*r**2
rhs = (1+q*s)*L*r*x  - L/2*(1 + 2*q*s + q*s**2)*r**2

diff1 = (lhs - mid).subs(x, (1+s)*r).subs(mu, q*L)
diff2 = (mid - rhs).subs(x, (1+s)*r).subs(mu, q*L)

print(simplify(diff1))
print(simplify(diff2))


0
0


In [4]:
# Proposition 3.4 (primitive belongs to f-dominant)
eta = 1 + 2*s + q*s**2

print(simplify(eta - (1 - q)*s - (1 + s)*(q*s + 1)))


lhs = (q*s + 1)*(xn - xs) - (q*s + 1)*(s + 1)/(1 - q)*(gn/L - q*(xn - xs))
rhs = (q*s + 1)/(1 - q)*((q*s + 1)*(xn - xs) - (s + 1)*gn/L)
print(simplify(lhs - rhs))


lhs = L/2*(xn - xs)**2 + s*(s + 1)/2/L*gn**2 + q*L*s/2/(1 - q)*(xn - xs - gn/L)**2\
    + (1 + s + q*s**2)*Qns - (1 + s + q*s**2)*(fn - fs)
rhs = L/2/(1 - q)*((q*s + 1)*(xn - xs) - (s + 1)*gn/L)**2
print(simplify(lhs - rhs))

0
0
0


### Section 3.3. Proofs of the Concatenation Theorems

In [5]:
# Lemma 3.9. Sufficient decrease
xn1 = xn - a/L*gn
D = q*s + s + 2 - (2*q*s + q + 1)*a
Cn = (s + 3*a - 2*a**2 - q*a**2*(s + 2 - a))/D
Cm = (2*s**2 + 3*s + a + q*s**2*(s + 2 - a))/D

Cpn1m = L/2*(xn1 - xs)**2 + (1 - q)*P2 + q*s*Qms - s*(fm - fs)\
    - L/2*(xm - xs)**2 - s*(s + 1)/2/L*gm**2 - q*L*s/2/(1 - q)*(xm - xs - gm/L)**2

Tn = (gn - L*(xn - xs))*(xn1 - xs - (q*s + 1)*(xm - xs)) - (1 - q)*(s*Qsn + P2)
Tm = (gm - L*(xm - xs))*(xn1 - xs - (q*s + 1)*(xm - xs)) - (1 - q)*(s*Qsm + P2)

rhs_a = (1 - q)*s*(fn - fm) + (a**2 - 2*a - s)/2/L*gn**2 - s*(s + 1)/2/L*gm**2\
    + (1 + q*s)*(gn*(xn - xm) - L/2*(xn - xm)**2)
print(simplify(Cpn1m + Tn - rhs_a))


rhs_b = (2 - (1 + q)*a)*(fn - fm) + a*(a - 1)/2/L*gn**2 - (s*(s + 2) + a)/2/L*gm**2\
    - (1 - q*a)*(gn*(xn - xm) - L/2*(xn - xm)**2)
print(simplify(Cpn1m + Tm - (a - 1)*Qnm - (1 - q*a)*Qmn - rhs_b))


cert_sd_lhs = (1 - q*a)*(Cpn1m + Tn) + (q*s + 1)*(Cpn1m + Tm - (a - 1)*Qnm - (1 - q*a)*Qmn)
cert_sd_rhs = D*(fn - fm - Cn*gn**2/2/L - Cm*gm**2/2/L)
cert_sd = cert_sd_lhs - cert_sd_rhs
print(simplify(cert_sd))

0
0
0


In [6]:
# Theorem 3.8 ConGP
sa, sb, sc = symbols('s_{a} s_{b} s_{c}')

sb = b - 2 + 2*(b - 1)**2*(q*sa + 1)**2/((q*sa + 1)**2*(q*sa**2 + 2*sa + 1) - q*(q*sa**2 + 2*sa + b)**2)
sb = simplify(sb)
sc = (sa + b + sb + q*sa*sb)/(1 - q*b)

Cn = (sb + 3*b - 2*b**2 - q*b**2*(sb + 2 - b))/(q*sb + sb + 2 - (2*q*sb + q + 1)*b)
Cm = (2*sb**2 + 3*sb + b + q*sb**2*(sb + 2 - b))/(q*sb + sb + 2 - (2*q*sb + q + 1)*b)

print(simplify(Cn + 2*sa + q*sa**2))
print(simplify(Cm - 2*sc - q*sc**2))

0
0


In [7]:
# Theorem 3.6 ConPP
sa, sb, sc = symbols('s_{a} s_{b} s_{c}')

xn1 = xn - a/L*gn
sa = a - 2 + 2*(a - 1)**2*(q*sb + 1)/(q*sb + sb + 2 - (2*q*sb + q + 1)*a)
sa = simplify(sa)
sc = (sa + a + sb + q*sa*sb)/(1 - q*a)

P3 = P1 + a*(q*sa + 1)/(1 - q*a)*Qns + (q*sa + 1)/(1 - q*a)*P2

Cp0m = L/2*(x0 - xs)**2 + (1 - q)*P3 + q*sc*Qms - sc*(fm - fs)\
    - L/2*(xm - xs)**2 - sc*(sc + 1)/2/L*gm**2 - q*L*sc/2/(1 - q)*(xm - xs - gm/L)**2
Cp0n = L/2*(x0 - xs)**2 + (1 - q)*P1 + q*sa*Qns - sa*(fn - fs)\
    - L/2*(xn - xs)**2 - sa*(sa + 1)/2/L*gn**2 - q*L*sa/2/(1 - q)*(xn - xs - gn/L)**2
Cpn1m = L/2*(xn1 - xs)**2 + (1 - q)*P2 + q*sb*Qms - sb*(fm - fs)\
    - L/2*(xm - xs)**2 - sb*(sb + 1)/2/L*gm**2 - q*L*sb/2/(1 - q)*(xm - xs - gm/L)**2

Cn = (sb + 3*a - 2*a**2 - q*a**2*(sb + 2 - a))/(q*sb + sb + 2 - (2*q*sb + q + 1)*a)
Cm = (2*sb**2 + 3*sb + a + q*sb**2*(sb + 2 - a))/(q*sb + sb + 2 - (2*q*sb + q + 1)*a)

cert_p = Cp0m - (Cp0n + (q*sa + 1)/(1 - q*a)*Cpn1m\
    + (1 - q)*(sa + a)/(1 - q*a)*(fn - fm - Cn*gn**2/2/L - Cm*gm**2/2/L))
print(simplify(cert_p))

0


In [8]:
# Theorem 3.7 ConPF
sa, sb, sc = symbols('s_{a} s_{b} s_{c}')

sa = b - 2 + 2*(b - 1)**2*(q*sb + 1)**2/((q*sb + 1)**2*(q*sb**2 + 2*sb + 1) - q*(q*sb**2 + 2*sb + b)**2)
sa = simplify(sa)

# quantities from Lemma 3.11
gamma = (sa + 2 - q*b)*(q*sa + 1)/(1 - q)/(1 - q*b)
v = (sa + 1)**2/(b - 1)**2
w = (sa + 1)*(q*sa + 2 - q*b)/(1 - q*b)/(1 + 2*sb + q*sb**2)

sc = (sa + b + sb + q*sa*sb)/(1 - q*b)
eta_b = 1 + 2*sb + q*sb**2
eta_c = 1 + 2*sc + q*sc**2
euc = sa + gamma + (v - w)*eub

P3 = P1 + gamma*Qns + (v - w)*P2
R1 = x0 - xs - (q*sa + 1)*(xn - xs)
R3 = R1 + gamma*(gn/L - q*(xn - xs)) + (v - w)*R2

xn1 = xn - b/L*gn

Cf0m = L/2*(x0 - xs)**2 + (1 - q)*P3 + (eta_c - (1 - q)*euc)*Qms - eta_c*(fm - fs)\
    - (1 - q)*L/2/(1 + q*sc)**2*(x0 - xs - R3 - (eta_c - (1 - q)*euc)/(1 - q)*(gm/L - q*(xm - xs)))**2
Cp0n = L/2*(x0 - xs)**2 + (1 - q)*P1 + q*sa*Qns - sa*(fn - fs)\
    - L/2*(xn - xs)**2 - sa*(sa + 1)/2/L*gn**2 - q*L*sa/2/(1 - q)*(xn - xs - gn/L)**2
Cfn1m = L/2*(xn1 - xs)**2 + (1 - q)*P2 + (eta_b - (1 - q)*eub)*Qms - eta_b*(fm - fs)\
    - (1 - q)*L/2/(1 + q*sb)**2*(xn1 - xs - R2 - (eta_b - (1 - q)*eub)/(1 - q)*(gm/L - q*(xm - xs)))**2
T = (gn - L*(xn - xs))*R2 - (1 - q)*(eub*Qsn + P2)

cert_f = Cf0m - (Cp0n + v*Cfn1m + w*T - w*(eta_b - (1 - q)*eub)*Qmn)

print(simplify(cert_f))   # this may take some seconds


v, w = symbols('v w')

0


### Section 5.1. The Silver Schedule

In [9]:
# Proposition 5.1 (i)
A1 = -q*s*Qns + s*(fn - fs) + L/2*(xn - xs)**2 + s*(s + 1)/2/L*gn**2\
    + q*L*s/2/(1 - q)*(xn - xs - gn/L)**2

cert_x_lhs = A1 + ((s + 1 - q)*(2*q*s + q + 1) + q**2*(s + 1) + (q*s + 1))/(1 - q)*Qns\
    + 2*(s + 1)*(q*s + 1)/(1 - q)*Qsn
cert_x_rhs = ((2*q*s + q + 1)/(1 - q))**2*L/2*(xn - (q*s + s + 2)/(2*q*s + q + 1)*gn/L - xs)**2
cert_x = cert_x_lhs - cert_x_rhs

print(simplify(cert_x))

0


In [10]:
# Proposition 5.1 (ii)
alpha2 = (q*s + s + 2)/(2*q*s + q + 1)
D = q*s + s + 2 - (2*q*s + q + 1)*a
Nn = s + 3*a - 2*a**2 - q*a**2*(s + 2 - a)
Nm = 2*s**2 + 3*s + a + q*s**2*(s + 2 - a)

print(simplify(D.subs(a,alpha2)))
print(simplify(-Nn.subs(a,alpha2) - 2*(1 - q)**2*(s + 1)**2*(q*s + 1)**2/(2*q*s + q + 1)**3))
print(simplify(Nm.subs(a,alpha2) - 2*(s + 1)**2*(q*s + 1)**2/(2*q*s + q + 1)))

# sharpness
print(simplify((1 - q*alpha2)/(1 + q*s) - (1 - q)/(2*q*s + q + 1)))

0
0
0
0


### Appendix A.1. Lemmas of Algebraic Inequalities

In [11]:
# Lemma A.1. Characterization of varphi

Phi = (1 - q)*h**2 + (q*(2*x*y + x + y + 2) + x + y)*h - ((1 + q)*x*y + 2*(x + y + 1))
disc = (q*(2*x*y + x + y + 2) + x + y)**2 + 4*((1 + q)*x*y + 2*(x + y + 1))*(1 - q)

# show that one of the roots of Phi is varphi
print(simplify(disc/(1 - q)**2 - ((x + y + 2 + 2*q/(1 - q)*(x + 1)*(y + 1))**2 + 4*(x + 1)*(y + 1))))
print(simplify((q*(2*x*y + x + y + 2) + x + y)/(1 - q) - (x + y + 2*q/(1 - q)*(x + 1)*(y + 1))))

a1 = 1 + q*(1 - q)*(x + 1)*(y + 1)/(q**2*(x + 1)*(y + 1) + (q*x + 1)*(q*y + 1))
a2 = (q*x + x + 2)/(2*q*x + q + 1)
values = [1, a1, a2, x + 2]
expected = [
    -(1 - q)*(x + 1)*(y + 1),
    -(1 - q)**3*(x + 1)*(y + 1)*(q*x + 1)*(q*y + 1)/(q**2*(x + 1)*(y + 1) + (q*x + 1)*(q*y + 1))**2,
    2*(1 - q)**2*(x + 1)**2*(q*x + 1)/(2*q*x + q + 1)**2,
    2*(x + 1)**2*(q*y + 1),
]
for value, rhs in zip(values,expected):
    print(f'Phi({value}) =', Phi.subs(h,value).factor())
    assert simplify(Phi.subs(h,value) - rhs) == 0

print('='*50)
print(simplify(x + 2 - a2 - 2*q*(x + 1)**2/(2*q*x + q + 1)))
print(simplify(1/q - a2 - (1 - q)*(q*x + 1)/q/(2*q*x + q + 1)))
print(simplify(a2 - (1 + (1 - q)*(x + 1)/(2*q*x + q + 1))))

0
0
Phi(1) = (q - 1)*(x + 1)*(y + 1)
Phi(q*(1 - q)*(x + 1)*(y + 1)/(q**2*(x + 1)*(y + 1) + (q*x + 1)*(q*y + 1)) + 1) = (q - 1)**3*(x + 1)*(y + 1)*(q*x + 1)*(q*y + 1)/(2*q**2*x*y + q**2*x + q**2*y + q**2 + q*x + q*y + 1)**2
Phi((q*x + x + 2)/(2*q*x + q + 1)) = 2*(q - 1)**2*(x + 1)**2*(q*x + 1)/(2*q*x + q + 1)**2
Phi(x + 2) = 2*(x + 1)**2*(q*y + 1)
0
0
0


In [12]:
# Lemma A.2 Characterization of psi
Psi = (h - 1)**2*(2 + q*(x - h)) - (q*y**2 + 2*y + 1)*(q*x + x + 2 - (2*q*x + q + 1)*h)

# show that Psi0 is the following quadratic polynomial
Psi0 = 2*h**2 + (2*y - 3)*h - 2*x*y - x - 4*y
print('Psi =', Psi.factor())
print(simplify(Psi.subs(q,0) - Psi0))
print('='*80)

a2 = (q*x + x + 2)/(2*q*x + q + 1)
values = [1,a2,x + 2,1/q]
expected = [
    -(1 - q)*(x + 1)*(q*y**2 + 2*y + 1),
    2*(1 - q)**2*(x + 1)**2*(q*x + 1)**2/(2*q*x + q + 1)**3,
    2*(x + 1)**2*(q*y + 1)**2,
    (1 - q)*(q*x + 1)*(q*y + 1)**2/q**2,
]
for value, rhs in zip(values,expected):
    print(f'Psi({value}) =', Psi.subs(h,value).factor())
    assert simplify(Psi.subs(h,value) - rhs) == 0

Psi =

 -h**3*q + h**2*q*x + 2*h**2*q + 2*h**2 + 2*h*q**2*x*y**2 + h*q**2*y**2 + 4*h*q*x*y + h*q*y**2 + 2*h*q*y + 2*h*y - 3*h - q**2*x*y**2 - q*x*y**2 - 2*q*x*y - 2*q*y**2 - 2*x*y - x - 4*y
0
Psi(1) = (q - 1)*(x + 1)*(q*y**2 + 2*y + 1)
Psi((q*x + x + 2)/(2*q*x + q + 1)) = 2*(q - 1)**2*(x + 1)**2*(q*x + 1)**2/(2*q*x + q + 1)**3
Psi(x + 2) = 2*(x + 1)**2*(q*y + 1)**2
Psi(1/q) = -(q - 1)*(q*x + 1)*(q*y + 1)**2/q**2


In [13]:
# Lemma A.3. Equivalent forms of varphi and psi
Phi = (1 - q)*h**2 + (q*(2*x*y + x + y + 2) + x + y)*h - ((1 + q)*x*y + 2*(x + y + 1))
Psi = (h - 1)**2*(2 + q*(x - h)) - (q*y**2 + 2*y + 1)*(q*x + x + 2 - (2*q*x + q + 1)*h)

Dphi = q*y + y + 2 - (2*q*y + q + 1)*a
Dpsi = (q*y + 1)**2*(q*y**2 + 2*y + 1) - q*(q*y**2 + 2*y + b)**2
print(simplify(Phi.subs(h,a) + Dphi*(x + 2 - a) - 2*(a - 1)**2*(q*y + 1)))
print(simplify(Psi.subs(h,b) + Dpsi*(x + 2 - b) - 2*(b - 1)**2*(q*y + 1)**2))

x_phi = a - 2 + 2*(a - 1)**2*(q*y + 1)/Dphi
x_psi = b - 2 + 2*(b - 1)**2*(q*y + 1)**2/Dpsi
print(simplify(Phi.subs({h:a,x:x_phi})))
print(simplify(Psi.subs({h:b,x:x_psi})))

0
0
0
0


In [14]:
# Lemma A.4. Monotonicity of varphi and psi
Phi = (1 - q)*h**2 + (q*(2*x*y + x + y + 2) + x + y)*h - ((1 + q)*x*y + 2*(x + y + 1))
Psi = (h - 1)**2*(2 + q*(x - h)) - (q*y**2 + 2*y + 1)*(q*x + x + 2 - (2*q*x + q + 1)*h)

print(simplify(Phi.diff(x) - ((2*q*y + q + 1)*h - (q*y + y + 2))))
print(simplify(Phi.diff(y) - ((2*q*x + q + 1)*h - (q*x + x + 2))))


Psidiffx = (q*Psi - 2*(1 - q*h)**2*(q*y**2 + 2*y + 1))/((q*x + 1) + (1 - q*h))
Psidiffy = -2*(q*y + 1)*(q*x + x + 2 - (2*q*x + q + 1)*h)
print(simplify(Psidiffx - Psi.diff(x)))
print(simplify(Psidiffy - Psi.diff(y)))

0
0
0
0


In [15]:
# Lemma A.5. Comparison of varphi and psi
Phi = (1 - q)*h**2 + (q*(2*x*y + x + y + 2) + x + y)*h - ((1 + q)*x*y + 2*(x + y + 1))
Psi = (h - 1)**2*(2 + q*(x - h)) - (q*y**2 + 2*y + 1)*(q*x + x + 2 - (2*q*x + q + 1)*h)

print(simplify(Psi - (q*y + 1)*Phi + (1 - q*h)*(x + 2 - h)*((1 - q)*y + h - 1)))


a1 = 1 + q*(1 - q)*(x + 1)*(y + 1)/(q**2*(x + 1)*(y + 1) + (q*x + 1)*(q*y + 1))
a2 = (q*x + x + 2)/(2*q*x + q + 1)
cq = 2/(1 + q)
lhs = (q*x + 1)*(q*cq*y + 1)/(1 - q*a2) - 1
rhs = cq*((q*x + 1)*(q*y + 1)/(1 - q*a1) - 1)
print(simplify(lhs - rhs))

0
0


In [16]:
# Proof of Lemma 3.11

sa, sb, sc = symbols('s_{a} s_{b} s_{c}')
sa = b - 2 + 2*(b - 1)**2*(q*sb + 1)**2/((q*sb + 1)**2*(q*sb**2 + 2*sb + 1) - q*(q*sb**2 + 2*sb + b)**2)
sa = simplify(sa)

# quantities of Lemma 3.11
gamma = (sa + 2 - q*b)*(q*sa + 1)/(1 - q)/(1 - q*b)
v = (sa + 1)**2/(b - 1)**2
w = (sa + 1)*(q*sa + 2 - q*b)/(1 - q*b)/(1 + 2*sb + q*sb**2)
sc = (sa + b + sb + q*sa*sb)/(1 - q*b)
euc = sa + gamma + (v - w)*eub

print(simplify(v - w - (sa + 1)*(1 + q*sa)/(b - 1)/(1 - q*b)))
print(simplify(1 + 2*sc + q*sc**2 - (1 - q)*(sa + gamma) - (v - w)*(1 + 2*sb + q*sb**2)))

v, w = symbols('v w')

0
0


### Appendix A.2. Huber-type Equality Examples

In [17]:
# Proof of Lemma 3.1
sk, sk1 = symbols('sigma_k sigma_k1')

sk1 = (sk + h)/(1 - q*h)
print(simplify(sk + h*(1 + q*sk1) - sk1))

xk = (1 - (1 - q)*sk/s)/(1 + q*sk)*x0
xk1 = (1 - (1 - q)*sk1/s)/(1 + q*sk1)*x0

print(simplify(xk1/x0 - 1/s - (s - (1 + sk1))/(s*(1 + q*sk1))))

0
0


### Appendix B. Auxiliary Results for Section 4

In [18]:
# Lemma B.1
root = sqrt(x**2 + 2*x + 5)
G = (x**2 + 1 + (x - 1)*root)/2

# Rationalization and derivative used in items (i) and (ii)
print((G - x**2 - 2*(x - 1)/(root + x + 1)).simplify())
print(((G/x**2).diff(x) - (5 - x - root)/(x**3*root)).simplify())
print((G/x**2).subs(x,Rational(5,3)) - Rational(27,25))
print((5 - x - root).subs(x,Rational(5,3)))

# Item (iii)
print(((x - 1 + 2*sqrt(2))**2 - (x**2 + 2*x + 5) - 4*(sqrt(2) - 1)*(x - 1)).expand())
print((G - 1 - (x - 1)/2*(x + 1 + root)).simplify())

# Item (iv)
print((G.diff(x) - x - (x**2 + x + 2)/root).simplify())
print(simplify((x**2 + x + 2)**2 - x**2*(x**2 + 2*x + 5) - 4*(x + 1)))

0
0
0
0
0
0
0
0


In [19]:
# Lemma B.3. (i)
disc = (x*y + 3)**2 - 4*(x + 1)*(y + 1)
print(simplify(disc - ((x*y - 1)**2 + 4*(x - 1)*(y - 1))))
print(simplify((x*y + 1)**2 - disc - 4*(x + y - 1)))

xr = r - 1/(r + 1)
ys = s - 1/(s + 1)

# check sqrt(disc) == root with the parametrization
root = (r**2 - 2)*(r**2 + 2*r + 2)/(r + 1)**2
print(simplify(disc.subs(y,x).subs(x,xr) - root**2))

print(simplify(((x**2 + 1 + root)/2).subs(x,xr) - (r**2 - 1)))

lhs = ((r**2 - 1)*(s**2 - 1) + x + y - 1)**2 - (x*y + 1)**2*(r**2 - 1)*(s**2 - 1)
lhs = lhs.subs({x:xr,y:ys})
Prs = (r**2 - 2)*(s**2 - 2)*(r**2 - 1)*(s**2 - 1) + (r*s + r + s + 2)**2
rhs = (r - s)**2*Prs/(r + 1)**2/(s + 1)**2

print(simplify(lhs - rhs))

0
0
0
0
0


In [20]:
# Lemma B.3 (ii)
x, y, z, u, v, w, r, s = symbols('x y z u v w r s')
e3 = w**2 - (x*y + 1)*w + x + y - 1
e4 = z**2 - (u*v + 1)*z + u + v - 1
e5 = z**2 - (w**2 + 1)*z + 2*w - 1

A = u*v - w**2
B = u + v - 2*w
R = B**2 - (w**2 + 1)*A*B + (2*w - 1)*A**2
print(simplify(resultant(e4,e5,z) - R))


subs_rs = {x: r - 1/(r + 1), y: s - 1/(s + 1), u: r**2 - 1, v: s**2 - 1}
E3 = simplify((r + 1)*(s + 1)*e3.subs(subs_rs))
rhs = E3*(E3 + 2*(r + s)*(w - 1)*(r*s - w - 1)) - (r**2 - 2)*(s**2 - 2)*(w - 1)**2*(r*s - w - 1)**2
print(simplify(R.subs(subs_rs) - rhs))

# strictness
print(simplify(e3.subs(w,1) + (x - 1)*(y - 1)))
print(simplify(e3.subs(subs_rs).subs(w, r*s - 1) - r*s*(r - s)**2/(r + 1)/(s + 1)))

0
0
0
0


### Appendix C. Proofs for Section 5

In [21]:
# Proof of Corollary 5.5.
lhs = ((1 - q*h)**2*(2 - h)/(2 - h*(1 + q)) - 1)
rhs = -q*h*((2-h)*(2-q*h) - 1)/(2 - h*(1 + q))

print(simplify(lhs - rhs))

0
